# Phase 6 — Abstractive Legal Summarization — Model 6 — LoRA

Run this notebook from top to bottom in a fresh kernel with the matching foundation checkpoint and tokenizer. The experiment is already selected in the configuration cell.

**Source:** `phase6/capstone-p6-all-models-all-methods.ipynb` at commit `7e7c8a9f1605c5f9836a3cb2cb7c28004343f5f3`. Model/training/evaluation/submission implementations are copied unchanged; only configuration selections and output directories differ. Saved outputs and execution counts were cleared during organization.

**Outputs:** use a separate Kaggle session or working directory for each run. The original submission filenames are preserved, so some ZIP files at the working-directory root can be overwritten by another run.

## Original implementation notes

# Phase 6 — Abstractive legal summarization
## Your own models (Model 4 / Model 6 / Model 7) as encoder + autoregressive decoder | Modes: **lora**, **lorma**, **full**, **knowledge_distillation**

Same methodology as your original Phase 6 notebook, but the backbone is only ever one of **your own pre-trained models**. Pick the architecture with `MODEL_OPTION` in the config cell: `4` = Model 4 (`EfficientLegalLM`, 16 layers, 768d), `6` = Model 6 (`EfficientLegalLM`, 20 layers, 1,024d) or `7` = Model 7 (12-layer BERT-style stack, 768d, uncased WordPiece). Only the selected model is loaded.

**How the encoder–decoder is built from your model (the same pre-trained weights initialise both halves):**
* **Model 7:** encoder = the Model 7 `BertModel` (bidirectional); decoder = a second copy of Model 7 as a causal `BertLMHeadModel` with newly initialised cross-attention, wrapped in Hugging Face `EncoderDecoderModel` (generation = HF beam search, exactly like the original notebook). The decoder's LM head is initialised from the prediction head stored in your checkpoint when it is there.
* **Models 4/6:** encoder = the model run **bidirectionally** (as in your Phase 3/4 notebooks); decoder = a second copy run **causally** exactly as pre-trained (alternating sliding-window / full layers), with a **newly initialised cross-attention block inserted after the self-attention of every decoder layer**. The LM head is the tied token embedding. These models have no `[CLS]`/`[SEP]`, so the tokenizer's BOS (or `<eos>` if it has none) starts decoding and `<eos>` ends it. Generation uses a built-in beam search (KV-cached, `no_repeat_ngram`, `min_new_tokens`, length penalty) because Hugging Face `generate` does not know this architecture.

**Unchanged from the original:** windows span the document and their token states are concatenated for decoder cross-attention; one document = one training example; document-disjoint validation split; checkpoint selection by generated-summary BERTScore F1 (+ ROUGE-1/L); the four modes (adapters and the new cross-attention always train; `full` trains everything; LoRMA keeps the Phase 3/4/5 multiplicative output-space definition); **sequence-level** knowledge distillation (a fully fine-tuned teacher writes pseudo-summaries for training documents; a 6-layer-encoder / 6-layer-decoder student, initialised from evenly spaced pre-trained layers, learns from references + teacher sequences; teacher and student never share the GPU).

**Differences you should know about:** (1) learning rates default to the Phase 3/4 values for your models (full 5e-5, adapters 3e-4; new cross-attention stays 1e-4); (2) checkpoints store only the trainable tensors, because everything else is re-created from your pre-trained checkpoint when restoring (smaller files, same result); (3) `'kd'` is accepted as an alias for `'knowledge_distillation'`.

Run top to bottom on Kaggle with a GPU and Internet enabled (Internet is needed for the HF data and the BERTScore model). Attach the checkpoint dataset (`best.pt`) of the selected model and its tokenizer dataset. This notebook has only been syntax-checked, not executed.

## 1. Dependencies and imports

In [ ]:
%pip install -q transformers huggingface_hub numpy rouge-score bert-score safetensors

In [ ]:
import os, json, math, random, gc, hashlib, shutil, zipfile, copy, glob
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from tqdm.auto import tqdm
from transformers import (AutoTokenizer, EncoderDecoderModel, BertConfig, BertModel, BertLMHeadModel,
                          PreTrainedTokenizerFast, BertTokenizerFast, get_linear_schedule_with_warmup)
from transformers.modeling_outputs import BaseModelOutput
from huggingface_hub import hf_hub_download
from rouge_score import rouge_scorer
from bert_score import score as bert_score

## 2. Configuration

In [ ]:
MODEL_OPTION = 6        # <<< CHOOSE which of your models: 4 | 6 | 7
RUN_METHODS = ['lora'] # any subset of: lora, lorma, full, knowledge_distillation ('kd' is accepted as an alias)
CKPT_PATH = None        # set to force a specific best.pt (otherwise auto-found under /kaggle/input)
TOK_DIR = None          # set to force a tokenizer folder (otherwise auto-found under /kaggle/input)
DATA_ROOT = None # optional directory containing Train/train.json and Test/test.json
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
HF_TOKEN = os.environ.get('HF_TOKEN')
OUTPUT_DIR = Path('/kaggle/working/summ_p6_model6_lora' if Path('/kaggle/working').exists() else './summ_p6_model6_lora')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED=42; SMOKE=False; VAL_FRAC=0.1
MAX_LEN=256; CHUNKS=4; STRIDE=192
REPRESENTATION='uniform' # uniform or head_tail (more budget at boundaries)
MAX_TARGET=256; MAX_NEW_TOKENS=255; MIN_NEW_TOKENS=24
BEAMS=4; LENGTH_PENALTY=1.0; NO_REPEAT_NGRAM=3
BATCH_SIZE=1; GRAD_ACCUM=8; EPOCHS=6; PATIENCE=2
FULL_LR=5e-5; ADAPTER_LR=3e-4; NEW_LR=1e-4   # full / adapter values from your Phase 3-4 runs of these models
RANK=8; ALPHA=16.; WEIGHT_DECAY=0.01; WARMUP_FRAC=0.1
GRAD_CHECKPOINT=True
BIDIR_WINDOW=256        # Models 4/6: width of the sliding-window encoder layers when run bidirectionally (as Phase 3/4)
STUDENT_LAYERS=6; KD_HARD_WEIGHT=0.5
# Local metric proxy; competition BERTScore model/settings are not specified.
BERTSCORE_MODEL='distilbert-base-uncased'; BERTSCORE_LAYERS=5

# ===================== architecture of the selected model =====================
assert MODEL_OPTION in (4, 6, 7), 'MODEL_OPTION must be 4, 6 or 7'
if MODEL_OPTION in (4, 6):
    ARCH = 'lm'                 # EfficientLegalLM (RMSNorm, RoPE, GQA, SwiGLU, alternating window/full layers)
    if MODEL_OPTION == 4:
        D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, D_FF, WINDOW_SIZE = 768, 16, 12, 2, 2048, 256
    else:
        D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, D_FF, WINDOW_SIZE = 1024, 20, 16, 4, 2432, 512
    HEAD_DIM, MAX_SEQ_LEN, ROPE_BASE = D_MODEL // N_HEADS, 1024, 10000.0
else:
    ARCH = 'bert'               # Model 7: BERT-base-style stack, prefix-LM pre-trained
    M7_CFG = dict(vocab_size=30522, hidden_size=768, num_hidden_layers=12, num_attention_heads=12,
                  intermediate_size=3072, hidden_act='gelu', max_position_embeddings=512, type_vocab_size=2,
                  layer_norm_eps=1e-12, hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1,
                  position_embedding_type='absolute')
    D_MODEL, MAX_SEQ_LEN = M7_CFG['hidden_size'], M7_CFG['max_position_embeddings']
    CKPT_KEY_PREFIX = ''        # optional extra prefix to strip from checkpoint keys (after '_orig_mod.' / 'module.' / 'bert.')
MODEL_NAME = f'Model {MODEL_OPTION}'

RUN_METHODS = ['knowledge_distillation' if m == 'kd' else m for m in RUN_METHODS]
assert RUN_METHODS and len(set(RUN_METHODS))==len(RUN_METHODS)
assert set(RUN_METHODS)<= {'lora','lorma','full','knowledge_distillation'}
assert REPRESENTATION in ('uniform','head_tail') and CHUNKS>=2
assert 0 < VAL_FRAC < 1 and 0 <= KD_HARD_WEIGHT <= 1
assert MAX_LEN <= MAX_SEQ_LEN and MAX_TARGET <= MAX_SEQ_LEN and MIN_NEW_TOKENS < MAX_NEW_TOKENS < MAX_SEQ_LEN
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP=DEVICE.type=='cuda'
def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
def cleanup():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
seed_all()
print(f'{MODEL_NAME} ({ARCH}) | methods {RUN_METHODS} | {DEVICE} | {OUTPUT_DIR}')

## 3. Locate the checkpoint and tokenizer of the selected model

In [ ]:
from transformers import PreTrainedTokenizerFast, BertTokenizerFast, AutoTokenizer

def _find(patterns, prefer=()):
    hits = sorted({h for p in patterns for h in glob.glob(p, recursive=True)})
    for key in prefer:
        pref = [h for h in hits if key in h.lower()]
        if pref:
            return pref[0], hits
    return (hits[0] if hits else None), hits

if CKPT_PATH is None:
    prefer = {4: ('m4',), 6: ('m6',), 7: ('m7', 'model7', 'model_7')}[MODEL_OPTION] + ('capstone',)
    CKPT_PATH, hits = _find(['/kaggle/input/**/best.pt'], prefer=prefer)
    print('best.pt candidates:', hits)
if TOK_DIR is None:
    if ARCH == 'lm':
        tj, hits = _find(['/kaggle/input/**/tokenizer.json'], prefer=('legal_tokenizer', 'tokenizer'))
    else:
        tj, hits = _find(['/kaggle/input/**/vocab.txt', '/kaggle/input/**/tokenizer.json'], prefer=('m7', 'model7', 'legal', 'tokenizer'))
    TOK_DIR = os.path.dirname(tj) if tj else None
    print('tokenizer candidates:', hits)
assert CKPT_PATH and os.path.isfile(CKPT_PATH), f'best.pt for {MODEL_NAME} not found - attach its checkpoint dataset or set CKPT_PATH'
assert TOK_DIR and os.path.isdir(TOK_DIR), f'tokenizer for {MODEL_NAME} not found - attach its tokenizer dataset or set TOK_DIR'
print(f'{MODEL_NAME}: make sure these belong to {MODEL_NAME}')
print('using checkpoint:', CKPT_PATH)
print('using tokenizer :', TOK_DIR)

if ARCH == 'lm':
    tokenizer = PreTrainedTokenizerFast.from_pretrained(TOK_DIR)
else:
    try:                                   # uncased WordPiece, 30,522 entries
        tokenizer = BertTokenizerFast.from_pretrained(TOK_DIR, do_lower_case=True)
    except Exception as e:
        print('BertTokenizerFast failed (', e, ') - falling back to AutoTokenizer')
        tokenizer = AutoTokenizer.from_pretrained(TOK_DIR)
tokenizer.model_max_length = 10 ** 9      # we window ourselves; silence the length warning
VOCAB_SIZE = len(tokenizer)
print(f'VOCAB_SIZE={VOCAB_SIZE} fast={tokenizer.is_fast}')

if ARCH == 'lm':
    PAD_L = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    SEP_L = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else PAD_L
    CLS_L, N_SPECIAL = None, 1             # window = tokens + <eos>
    def wrap(chunk): return list(chunk) + [SEP_L]
else:
    assert VOCAB_SIZE == M7_CFG['vocab_size'], f'tokenizer has {VOCAB_SIZE} entries but Model 7 was built for {M7_CFG["vocab_size"]}'
    assert None not in (tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id)
    PAD_L, SEP_L, CLS_L, N_SPECIAL = tokenizer.pad_token_id, tokenizer.sep_token_id, tokenizer.cls_token_id, 2
    def wrap(chunk): return [CLS_L] + list(chunk) + [SEP_L]     # window = [CLS] tokens [SEP]
    print('tokenizer lowercases:', tokenizer.tokenize('Section 302 IPC, The Supreme Court'))
print('PAD', PAD_L, 'SEP/EOS', SEP_L, 'CLS', CLS_L, '| special tokens per window:', N_SPECIAL)

In [ ]:
# decoder start / end tokens (Model 7: [CLS] / [SEP]; Models 4/6: BOS if the tokenizer has one, else <eos>; <eos> ends the summary)
EOS_ID = SEP_L
START_ID = CLS_L if ARCH == 'bert' else (tokenizer.bos_token_id if tokenizer.bos_token_id is not None else EOS_ID)
if ARCH == 'lm' and tokenizer.eos_token_id is None: print('WARNING: tokenizer has no <eos>; falling back to the pad id as end-of-summary token')
print('decoder START', START_ID, '| EOS', EOS_ID, '| PAD', PAD_L)

## 4. Phase 6 data and document-disjoint validation
No public dev file is assumed. Exact duplicate documents are grouped before splitting; conflicting duplicate IDs are rejected. Test references are never read.

In [ ]:
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN=UserSecretsClient().get_secret('HF')
    except Exception:
        pass

def read_split(name):
    rel = f"{'Train' if name == 'train' else 'Test'}/{name}.json"

    path = Path(DATA_ROOT)/rel if DATA_ROOT else hf_hub_download(
        repo_id=HF_DATASET_REPO,
        repo_type='dataset',
        filename=f'Phase_6_Summ/{rel}',
        token=HF_TOKEN
    )

    with open(path, encoding='utf-8') as f:
        raw = json.load(f)

    if isinstance(raw, dict):
        if isinstance(raw.get('data'), list):
            raw = raw['data']
        elif isinstance(raw.get('examples'), list):
            raw = raw['examples']
        else:
            raw = [
                dict(v, id=k) if isinstance(v, dict)
                else {'id': k, 'document': v}
                for k, v in raw.items()
            ]

    assert isinstance(raw, list) and raw

    rows = []
    seen = set()
    skipped = 0

    for r in raw:
        assert isinstance(r, dict) and 'id' in r and 'document' in r
        assert isinstance(r['id'], (str, int)) and not isinstance(r['id'], bool)
        assert str(r['id']) not in seen, f"Duplicate ID: {r['id']}"
        seen.add(str(r['id']))

        assert isinstance(r['document'], str) and r['document'].strip()

        item = {'id': r['id'], 'document': r['document']}

        if name == 'train':
            if not isinstance(r.get('summary'), str) or not r['summary'].strip():
                skipped += 1
                continue

            item['summary'] = r['summary']

        rows.append(item)

    print(f"{name}: Loaded {len(rows)}, skipped {skipped}")

    assert rows
    return rows
all_train=read_split('train'); test=read_split('test')
def doc_key(r): return hashlib.sha256(' '.join(r['document'].split()).encode()).hexdigest()
groups=sorted({doc_key(r) for r in all_train})
assert len(groups)>=2, 'Need at least two distinct training documents'
random.Random(SEED).shuffle(groups)
nval=max(1,min(len(groups)-1,round(len(groups)*VAL_FRAC)))
val_keys=set(groups[:nval])
train=[r for r in all_train if doc_key(r) not in val_keys]
dev=[r for r in all_train if doc_key(r) in val_keys]
assert not {doc_key(r) for r in train}&{doc_key(r) for r in dev}
if SMOKE:
    train=train[:8]; dev=dev[:2]; test=test[:3]; EPOCHS=1
print('train/dev/test:',len(train),len(dev),len(test))
with open(OUTPUT_DIR/'split_ids.json','w') as f:
    json.dump({'train':[r['id'] for r in train],'dev':[r['id'] for r in dev]},f)

## 5. Window selection and targets
Uniform selection includes both ends and evenly spaced windows. Head/tail reserves half the windows at the beginning and half at the end. References are capped at MAX_TARGET tokens; truncation counts below help choose a better budget. Windows are wrapped with the selected model's own special tokens (`[CLS] … [SEP]` for Model 7, `… <eos>` for Models 4/6). The decoder context is `MAX_SEQ_LEN` tokens (512 for Model 7, 1,024 for Models 4/6).

In [ ]:
WINDOW=MAX_LEN-N_SPECIAL  # window + the model's special tokens
assert 0<STRIDE<=WINDOW and MAX_LEN<=MAX_SEQ_LEN
assert MIN_NEW_TOKENS<MAX_NEW_TOKENS<MAX_SEQ_LEN and MAX_TARGET<=MAX_SEQ_LEN

def windows(text):
    ids=tokenizer(text,add_special_tokens=False,truncation=False)['input_ids']
    last=max(0,len(ids)-WINDOW)
    starts=sorted(set(list(range(0,last+1,STRIDE))+[last]))
    if len(starts)>CHUNKS:
        if REPRESENTATION=='uniform':
            starts=[starts[i] for i in np.linspace(0,len(starts)-1,CHUNKS).round().astype(int)]
        else:
            left=(CHUNKS+1)//2; starts=starts[:left]+starts[-(CHUNKS-left):]
    return [wrap(ids[s:s+WINDOW]) for s in starts]

def target(text):
    # Decoder starts with START_ID; labels contain text then SEP, without a leading CLS.
    return tokenizer(text,add_special_tokens=False,truncation=True,max_length=MAX_TARGET-1)['input_ids']+[EOS_ID]
class Cases(Dataset):
    def __init__(self,rows):
        self.rows=rows
        self.x=[windows(r['document']) for r in tqdm(rows,desc='Tokenize documents')]
    def __len__(self): return len(self.rows)
    def __getitem__(self,i): return self.x[i],self.rows[i]
def collate(items):
    c=max(len(x) for x,r in items); l=max(len(w) for x,r in items for w in x)
    ids=torch.full((len(items),c,l),PAD_L,dtype=torch.long)
    mask=torch.zeros_like(ids)
    for i,(ws,r) in enumerate(items):
        for j,w in enumerate(ws): ids[i,j,:len(w)]=torch.tensor(w); mask[i,j,:len(w)]=1
    rows=[r for x,r in items]
    return ids.to(DEVICE),mask.to(DEVICE),rows
def labels_for(rows,key='summary'):
    ys=[target(r[key]) for r in rows]
    y=torch.full((len(ys),max(map(len,ys))),-100,dtype=torch.long,device=DEVICE)
    for i,v in enumerate(ys): y[i,:len(v)]=torch.tensor(v,device=DEVICE)
    return y
def loader(ds,shuffle=False): return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,collate_fn=collate,num_workers=0)
train_ds=Cases(train); dev_ds=Cases(dev); test_ds=Cases(test)
print('Truncated training targets:',sum(len(tokenizer(r['summary'],add_special_tokens=False)['input_ids'])>MAX_TARGET-1 for r in train))

## 6. Your pre-trained backbones — definitions and checkpoint loading
Definitions are copied from your Phase 3 notebooks (`EfficientLegalLM` for Models 4/6; the BERT-style helpers for Model 7).

### 6a. Models 4 / 6 — `EfficientLegalLM` (from `capstone-p3-model-4-and-6.ipynb`)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).type_as(x) * self.weight


def precompute_rope_freqs(dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, freqs_cos, freqs_sin):
    x_r = x.reshape(*x.shape[:-1], -1, 2)
    x0, x1 = x_r[..., 0], x_r[..., 1]
    seq_len = x.shape[2]
    cos = freqs_cos[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    sin = freqs_sin[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    out0 = x0 * cos - x1 * sin
    out1 = x0 * sin + x1 * cos
    return torch.stack([out0, out1], dim=-1).reshape_as(x)


class SwiGLU_FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    B, H, T, D = x.shape
    return x[:, :, None, :, :].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)


class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.n_heads = n_heads; self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim; self.n_rep = n_heads // n_kv_heads
        self.use_window = use_window; self.window_size = window_size
        self.attn_p = dropout
        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, freqs_cos, freqs_sin)
        k = apply_rope(k, freqs_cos, freqs_sin)
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v) if use_cache else None
        dp = self.attn_p if self.training else 0.0
        if past_kv is not None:
            if self.use_window:
                k = k[:, :, -self.window_size:]; v = v[:, :, -self.window_size:]
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=dp)
        else:
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            if self.use_window and T > 1:
                row = torch.arange(T, device=x.device).unsqueeze(1)
                col = torch.arange(T, device=x.device).unsqueeze(0)
                mask = (col <= row) & (col > row - self.window_size)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp)
            else:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dp)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.out_proj(out)), new_kv


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAttention(d_model, n_heads, n_kv_heads, head_dim, dropout, use_window, window_size)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU_FFN(d_model, d_ff, dropout)

    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        attn_out, new_kv = self.attn(self.attn_norm(x), freqs_cos, freqs_sin, use_cache, past_kv)
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_kv


class EfficientLegalLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, head_dim, d_ff,
                 max_seq_len, rope_base=10000.0, dropout=0.0, window_size=256, use_gc=True):
        super().__init__()
        self.d_model = d_model; self.n_layers = n_layers; self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len; self.use_gc = use_gc
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout,
                             use_window=(i % 2 == 0), window_size=window_size)
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        fc, fs = precompute_rope_freqs(head_dim, max_seq_len, rope_base)
        self.register_buffer("freqs_cos", fc, persistent=False)
        self.register_buffer("freqs_sin", fs, persistent=False)

    def forward(self, input_ids, targets=None, use_cache=False, past_kvs=None):
        B, T = input_ids.shape
        x = self.emb_dropout(self.tok_emb(input_ids))
        past_len = past_kvs[0][0].shape[2] if past_kvs is not None else 0
        fc = self.freqs_cos[past_len:past_len + T]
        fs = self.freqs_sin[past_len:past_len + T]
        presents = [] if use_cache else None
        for i, layer in enumerate(self.layers):
            pkv = past_kvs[i] if past_kvs is not None else None
            x, kv = layer(x, fc, fs, use_cache, pkv)
            if use_cache:
                presents.append(kv)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits[:, :-1, :].contiguous().view(-1, self.vocab_size),
                                   targets[:, 1:].contiguous().view(-1), ignore_index=-100)
        return logits, loss, presents

### 6b. Model 7 — checkpoint helpers (from `capstone-p3-model7.ipynb`)

In [ ]:
from transformers import BertConfig, BertModel


def build_model7():
    """Model 7 backbone: embeddings (token + learned position + segment, LayerNorm) -> 12 post-LN layers -> pooler (unused)."""
    return BertModel(BertConfig(**M7_CFG), add_pooling_layer=True)


def _norm_key(k):
    for pre in ("_orig_mod.", "module."):
        while k.startswith(pre):
            k = k[len(pre):]
    if CKPT_KEY_PREFIX and k.startswith(CKPT_KEY_PREFIX):
        k = k[len(CKPT_KEY_PREFIX):]
    return k.replace(".LayerNorm.gamma", ".LayerNorm.weight").replace(".LayerNorm.beta", ".LayerNorm.bias")


def split_checkpoint(raw):
    """Return (encoder_state_dict, prediction_head_state_dict) from a Model 7 checkpoint of any common wrapping."""
    sd = raw
    if isinstance(raw, dict):
        for k in ("model_state_dict", "model", "state_dict", "global_state_dict", "theta_g"):
            if k in raw and isinstance(raw[k], dict):
                sd = raw[k]; break
    enc, head = {}, {}
    for k, v in sd.items():
        if not torch.is_tensor(v):
            continue
        k = _norm_key(k)
        if k.startswith("bert."):
            enc[k[len("bert."):]] = v
        elif k.startswith("cls."):
            head[k] = v
        else:
            enc[k] = v
    return enc, head


class PredHead:
    """BERT prediction (LM) head used only for the load sanity check: dense -> GELU -> LayerNorm -> tied decoder (+ bias)."""
    def __init__(self, head_sd, emb_w):
        g = lambda *names: next((head_sd[n] for n in names if n in head_sd), None)
        self.dw = g("cls.predictions.transform.dense.weight"); self.db = g("cls.predictions.transform.dense.bias")
        self.lw = g("cls.predictions.transform.LayerNorm.weight"); self.lb = g("cls.predictions.transform.LayerNorm.bias")
        self.bias = g("cls.predictions.bias", "cls.predictions.decoder.bias")
        self.emb_w = emb_w
        self.ok = None not in (self.dw, self.db, self.lw, self.lb)

    def __call__(self, h):
        d = F.gelu(F.linear(h, self.dw.to(h.device, h.dtype), self.db.to(h.device, h.dtype)))
        d = F.layer_norm(d, (D_MODEL,), self.lw.to(h.device, h.dtype), self.lb.to(h.device, h.dtype), M7_CFG["layer_norm_eps"])
        b = self.bias.to(h.device, h.dtype) if self.bias is not None else None
        return F.linear(d, self.emb_w.to(h.dtype), b)

### 6c. Load the selected model (same sanity checks as Phase 3)

In [ ]:
raw_ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
if ARCH == 'lm':
    ENC_SD = {(k[len('_orig_mod.'):] if k.startswith('_orig_mod.') else k): v for k, v in raw_ckpt['model_state_dict'].items()}
    print('checkpoint meta:', {k: raw_ckpt.get(k) for k in ('step', 'loss')})
    HEAD_SD = None
else:
    ENC_SD, HEAD_SD = split_checkpoint(raw_ckpt)
    print('checkpoint meta:', {k: raw_ckpt[k] for k in ('step', 'round', 'loss', 'val_loss') if isinstance(raw_ckpt, dict) and k in raw_ckpt})
del raw_ckpt

def build_backbone():
    # a fresh copy of the pre-trained model, weights loaded from ENC_SD (checked every time)
    if ARCH == 'lm':
        b = EfficientLegalLM(VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, HEAD_DIM, D_FF,
                             MAX_SEQ_LEN, ROPE_BASE, 0.0, WINDOW_SIZE, use_gc=False)
        missing, unexpected = b.load_state_dict(ENC_SD, strict=False)
        missing = [k for k in missing if k != 'lm_head.weight']          # tied to tok_emb.weight
    else:
        b = build_model7()
        missing, unexpected = b.load_state_dict(ENC_SD, strict=False)
        missing = [k for k in missing if not k.startswith('pooler.')]    # never trained by the continuation loss
        unexpected = [k for k in unexpected if k != 'embeddings.position_ids']
    assert not missing and not unexpected, f'checkpoint mismatch: missing={missing[:5]} unexpected={unexpected[:5]}'
    return b

# ---- sanity: a correctly loaded model is far better than chance on a training judgment ----
probe = build_backbone().to(DEVICE).eval()
_ids = tokenizer(train[0]['document'], add_special_tokens=False)['input_ids']
if ARCH == 'lm':
    ids0 = torch.tensor([_ids[:512]], device=DEVICE)
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
        lm_loss = probe(ids0, targets=ids0)[1].item()
    print(f'LM sanity loss on a training judgment: {lm_loss:.3f}  (expect roughly 2-4; chance = {math.log(VOCAB_SIZE):.1f})')
    assert lm_loss < 6.0, 'LM loss is too high - weights or tokenizer do not match'
else:
    ph = PredHead(dict(HEAD_SD), probe.embeddings.word_embeddings.weight.detach())
    if ph.ok:
        ids_ = _ids[:509]
        P = min(320, int(0.63 * len(ids_))); S = len(ids_) - P
        seq = [CLS_L] + ids_[:P] + [SEP_L] + ids_[P:] + [SEP_L]
        T, Lp = len(seq), P + 2
        idx = torch.arange(T, device=DEVICE)
        pm = ((idx[None, :] < Lp) | (idx[None, :] <= idx[:, None])).long()[None]     # prefix bidirectional, suffix causal
        tt = torch.tensor([[0] * Lp + [1] * (T - Lp)], device=DEVICE)
        try:
            with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
                hs = probe(input_ids=torch.tensor([seq], device=DEVICE), attention_mask=pm, token_type_ids=tt).last_hidden_state
                lg = ph(hs[:, Lp - 1:T - 1])
            tgt = torch.tensor([seq[Lp:]], device=DEVICE)
            lm_loss = F.cross_entropy(lg.float().reshape(-1, VOCAB_SIZE), tgt.reshape(-1)).item()
            print(f'continuation sanity loss ({S} suffix tokens): {lm_loss:.3f}  (chance = {math.log(VOCAB_SIZE):.2f}; a trained model is clearly lower)')
            assert lm_loss < 8.0, 'continuation loss is near chance - weights or tokenizer do not match'
        except AssertionError:
            raise
        except Exception as e:
            print('sanity check skipped (3-D attention mask not supported by this transformers version):', repr(e)[:200])
    else:
        print('prediction-head weights not in the checkpoint - sanity check skipped (only the encoder is needed)')
    del ph
del probe, _ids          # HEAD_SD is kept: Model 7's decoder LM head is initialised from it
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print(f'{MODEL_NAME} loaded OK: {sum(v.numel() for v in ENC_SD.values()) / 1e6:.1f}M tensor elements in the checkpoint')

## 7. Generative model and adaptation
`LoRALinear` / `LoRMALinear` / `add_adapters` are the original Phase 6 definitions (used for Model 7's `BertModel` encoder and decoder). `add_adapters_lm` applies the same two adapters to every linear of the Model 4/6 transformer blocks (q/k/v/out + SwiGLU gate/up/down). The **newly initialised cross-attention and the decoder LM head always train**; adapter modes freeze everything else; full mode trains all weights.

In [ ]:
class LoRALinear(nn.Module):
    # Conventional additive update: Wx + alpha/r * B(Ax).
    def __init__(self,base,rank,alpha):
        super().__init__();self.base=base
        self.A=nn.Parameter(torch.empty(rank,base.in_features))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5));self.scale=alpha/rank
    def forward(self,x):
        return self.base(x)+self.scale*F.linear(F.linear(x,self.A),self.B)

class LoRMALinear(nn.Module):
    # Same definition as Phase 3/4: multiplicative output-space adaptation (I + alpha/r * B A) W x, frozen W, B starts at zero.
    def __init__(self,base,rank,alpha):
        super().__init__();self.base=base
        self.A=nn.Parameter(torch.randn(rank,base.out_features)*0.02)
        self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        self.scale=alpha/rank
    def forward(self,x):
        h=self.base(x);return h+self.scale*F.linear(F.linear(h,self.A),self.B)

def add_adapters(encoder,method):
    assert method in ('lora','lorma')
    wrapper=LoRALinear if method=='lora' else LoRMALinear
    count=0
    for layer in encoder.encoder.layer:
        for parent,names in [(layer.attention.self,['query','key','value']),
                             (layer.attention.output,['dense']),
                             (layer.intermediate,['dense']),
                             (layer.output,['dense'])]:
            for name in names:
                linear=getattr(parent,name)
                assert isinstance(linear,nn.Linear),(name,type(linear))
                setattr(parent,name,wrapper(linear,RANK,ALPHA));count+=1
    return count

def add_adapters_lm(backbone,method):
    assert method in ('lora','lorma')
    wrapper=LoRALinear if method=='lora' else LoRMALinear
    count=0
    for blk in backbone.layers:
        for parent in (blk.attn,blk.ffn):
            for name,child in list(parent.named_children()):
                if isinstance(child,nn.Linear):
                    setattr(parent,name,wrapper(child,RANK,ALPHA));count+=1
    return count

### 7a. Models 4/6: encoder–decoder with cross-attention

In [ ]:
# ---- bidirectional masks for the encoder (every query may attend to itself, so padded rows never become NaN)
def build_masks(T, pad_mask, window):
    r = torch.arange(T, device=pad_mask.device).unsqueeze(1)
    c = torch.arange(T, device=pad_mask.device).unsqueeze(0)
    eye = torch.eye(T, dtype=torch.bool, device=pad_mask.device)[None, None]
    key = pad_mask[:, None, None, :]
    full = torch.ones(T, T, dtype=torch.bool, device=pad_mask.device)
    win = (c - r).abs() < window
    return (full[None, None] & key) | eye, (win[None, None] & key) | eye

class CrossAttention(nn.Module):
    # NEW parameters: pre-norm multi-head attention from decoder states to the concatenated window states (random init, always trained)
    def __init__(self, d, n_heads):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d // n_heads
        self.norm = RMSNorm(d)
        self.q_proj = nn.Linear(d, d, bias=False); self.k_proj = nn.Linear(d, d, bias=False)
        self.v_proj = nn.Linear(d, d, bias=False); self.out_proj = nn.Linear(d, d, bias=False)
        for l in (self.q_proj, self.k_proj, self.v_proj, self.out_proj): nn.init.normal_(l.weight, std=0.02)
    def kv(self, enc):
        B, S, _ = enc.shape
        k = self.k_proj(enc).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(enc).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        return k, v
    def forward(self, x, kv, kmask):
        B, T, _ = x.shape
        q = self.q_proj(self.norm(x)).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, kv[0], kv[1], attn_mask=kmask)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return x + self.out_proj(out)

def block_forward(blk, x, cos, sin, mask, xa=None, kv=None, kmask=None):
    # pre-trained self-attention -> (optional new cross-attention) -> pre-trained SwiGLU FFN
    a = blk.attn
    h = blk.attn_norm(x)
    B, T, _ = h.shape
    q = a.q_proj(h).view(B, T, a.n_heads, a.head_dim).transpose(1, 2)
    k = a.k_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    v = a.v_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)
    k = repeat_kv(k, a.n_rep)
    v = repeat_kv(v, a.n_rep)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    out = out.transpose(1, 2).contiguous().view(B, T, -1)
    x = x + a.out_proj(out)
    if xa is not None:
        x = xa(x, kv, kmask)
    return x + blk.ffn(blk.ffn_norm(x))

class LMEncoder(nn.Module):
    # (N, L) ids + pad mask -> (N, L, D); bidirectional, as in Phase 3/4
    def __init__(self, backbone, use_gc):
        super().__init__(); self.backbone, self.use_gc = backbone, use_gc
    def forward(self, ids, mask):
        m = self.backbone; T = ids.size(1)
        x = m.tok_emb(ids)
        cos, sin = m.freqs_cos[:T], m.freqs_sin[:T]
        full, win = build_masks(T, mask.bool(), BIDIR_WINDOW)
        for blk in m.layers:
            mk = win if blk.attn.use_window else full
            if self.use_gc and self.training:
                x = grad_checkpoint(block_forward, blk, x, cos, sin, mk, use_reentrant=False)
            else:
                x = block_forward(blk, x, cos, sin, mk)
        return m.final_norm(x)

class LMDecoder(nn.Module):
    # causal decoder exactly as pre-trained (window layers keep their causal sliding window) + new cross-attention per layer
    def __init__(self, backbone, use_gc):
        super().__init__(); self.backbone, self.use_gc = backbone, use_gc
        self.xattn = nn.ModuleList([CrossAttention(backbone.tok_emb.embedding_dim, backbone.layers[0].attn.n_heads) for _ in backbone.layers])
    def cross_kv(self, enc):
        return [xa.kv(enc) for xa in self.xattn]
    def forward(self, ids, enc_kv, enc_mask):            # training: full target sequence
        m = self.backbone; T = ids.size(1)
        x = m.tok_emb(ids)
        cos, sin = m.freqs_cos[:T], m.freqs_sin[:T]
        r = torch.arange(T, device=ids.device).unsqueeze(1); c = torch.arange(T, device=ids.device).unsqueeze(0)
        full = (c <= r)[None, None]
        win = ((c <= r) & (c > r - WINDOW_SIZE))[None, None]
        km = enc_mask.bool()[:, None, None, :]
        for blk, xa, kv in zip(m.layers, self.xattn, enc_kv):
            mk = win if blk.attn.use_window else full
            if self.use_gc and self.training:
                x = grad_checkpoint(block_forward, blk, x, cos, sin, mk, xa, kv, km, use_reentrant=False)
            else:
                x = block_forward(blk, x, cos, sin, mk, xa, kv, km)
        return m.lm_head(m.final_norm(x))
    def step(self, tok, pos, caches, enc_kv, km):          # generation: one new token per beam, self-attention KV cache
        m = self.backbone
        x = m.tok_emb(tok)
        cos, sin = m.freqs_cos[pos:pos + 1], m.freqs_sin[pos:pos + 1]
        new = []
        for blk, xa, kv, cache in zip(m.layers, self.xattn, enc_kv, caches):
            a = blk.attn
            h = blk.attn_norm(x); B = h.size(0)
            q = a.q_proj(h).view(B, 1, a.n_heads, a.head_dim).transpose(1, 2)
            k = a.k_proj(h).view(B, 1, a.n_kv_heads, a.head_dim).transpose(1, 2)
            v = a.v_proj(h).view(B, 1, a.n_kv_heads, a.head_dim).transpose(1, 2)
            q = apply_rope(q, cos, sin); k = apply_rope(k, cos, sin)
            if cache is not None:
                k = torch.cat([cache[0], k], dim=2); v = torch.cat([cache[1], v], dim=2)
            new.append((k, v))
            ka, va = (k[:, :, -a.window_size:], v[:, :, -a.window_size:]) if a.use_window else (k, v)
            out = F.scaled_dot_product_attention(q, repeat_kv(ka, a.n_rep), repeat_kv(va, a.n_rep))
            x = x + a.out_proj(out.transpose(1, 2).contiguous().view(B, 1, -1))
            x = xa(x, kv, km)
            x = x + blk.ffn(blk.ffn_norm(x))
        return m.lm_head(m.final_norm(x))[:, -1], new

def banned_ngram_tokens(gen, n):
    if n <= 0 or len(gen) < n: return []
    prefix = tuple(gen[len(gen) - (n - 1):]) if n > 1 else ()
    return [gen[i + n - 1] for i in range(len(gen) - n + 1) if n == 1 or tuple(gen[i:i + n - 1]) == prefix]

class SummLM(nn.Module):
    def __init__(self, enc_backbone, dec_backbone, use_gc):
        super().__init__()
        self.encoder = LMEncoder(enc_backbone, use_gc); self.decoder = LMDecoder(dec_backbone, use_gc)
    def encode(self, x, mask):
        b, c, l = x.shape
        # Encode windows sequentially to limit peak attention memory.
        states = [self.encoder(x[:, j], mask[:, j]) for j in range(c)]
        return torch.cat(states, dim=1), mask.reshape(b, c * l)
    def loss(self, enc, mask, y):
        prev = y[:, :-1].masked_fill(y[:, :-1] == -100, PAD_L)
        dec_in = torch.cat([torch.full((y.size(0), 1), START_ID, dtype=torch.long, device=y.device), prev], dim=1)
        logits = self.decoder(dec_in, self.decoder.cross_kv(enc), mask)
        return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=-100)
    @torch.no_grad()
    def generate(self, enc, mask):
        return [self.beam_search(enc[i:i + 1], mask[i:i + 1]) for i in range(enc.size(0))]
    @torch.no_grad()
    def beam_search(self, enc, mask):
        dev = enc.device; B = BEAMS; L = len(self.decoder.xattn)
        kv = [(k.expand(B, -1, -1, -1).contiguous(), v.expand(B, -1, -1, -1).contiguous()) for k, v in self.decoder.cross_kv(enc)]
        km = mask.bool().expand(B, -1)[:, None, None, :]
        tok = torch.full((B, 1), START_ID, dtype=torch.long, device=dev)
        scores = torch.zeros(B, device=dev); scores[1:] = -1e9      # only beam 0 is live at the first step
        caches = [None] * L; gens = [[] for _ in range(B)]; finished = []
        for step in range(MAX_NEW_TOKENS):
            logits, caches = self.decoder.step(tok, step, caches, kv, km)
            logp = F.log_softmax(logits.float(), dim=-1); V = logp.size(-1)
            if step < MIN_NEW_TOKENS: logp[:, EOS_ID] = -float('inf')
            if NO_REPEAT_NGRAM > 0:
                for b in range(B):
                    for t in banned_ngram_tokens(gens[b], NO_REPEAT_NGRAM): logp[b, t] = -float('inf')
            top_s, top_i = (scores[:, None] + logp).view(-1).topk(2 * B)
            keep = []
            for rank, (s, bi, t) in enumerate(zip(top_s.tolist(), (top_i // V).tolist(), (top_i % V).tolist())):
                if t == EOS_ID:
                    if rank < B: finished.append((s / ((step + 1) ** LENGTH_PENALTY), gens[bi]))
                else:
                    keep.append((s, bi, t))
                if len(keep) == B: break
            if len(finished) >= B: break                            # early stopping, as HF early_stopping=True
            idx = torch.tensor([bi for _, bi, _ in keep], device=dev)
            caches = [(k[idx], v[idx]) for k, v in caches]
            gens = [gens[bi] + [t] for _, bi, t in keep]
            scores = torch.tensor([s for s, _, _ in keep], device=dev)
            tok = torch.tensor([[t] for _, _, t in keep], dtype=torch.long, device=dev)
        else:
            for b in range(B): finished.append((scores[b].item() / (MAX_NEW_TOKENS ** LENGTH_PENALTY), gens[b]))
        return max(finished, key=lambda z: z[0])[1]

### 7b. Model 7: Hugging Face `EncoderDecoderModel`

In [ ]:
def build_m7_decoder():
    # Model 7 as a causal decoder: same weights, + NEW cross-attention; LM head initialised from your checkpoint when available
    cfg = BertConfig(**{**M7_CFG, 'is_decoder': True, 'add_cross_attention': True})
    dec = BertLMHeadModel(cfg)
    missing, unexpected = dec.bert.load_state_dict(ENC_SD, strict=False)
    missing = [k for k in missing if 'crossattention' not in k and not k.startswith('pooler.')]
    unexpected = [k for k in unexpected if k != 'embeddings.position_ids' and not k.startswith('pooler.')]
    assert not missing and not unexpected, f'decoder checkpoint mismatch: missing={missing[:5]} unexpected={unexpected[:5]}'
    if HEAD_SD:
        hs = {k[len('cls.'):]: v for k, v in HEAD_SD.items() if k.startswith('cls.') and 'predictions.decoder' not in k}
        m2, u2 = dec.cls.load_state_dict(hs, strict=False)
        print('decoder LM head initialised from the checkpoint | not in checkpoint:', m2, '| unused:', u2)
    else:
        print('no prediction head in the checkpoint: decoder LM transform is randomly initialised (output layer stays tied to the embeddings)')
    return dec

class SummBert(nn.Module):
    def __init__(self, hf):
        super().__init__(); self.hf = hf
    def encode(self, x, mask):
        b, c, l = x.shape
        # Encode windows sequentially to limit peak attention memory.
        states = [self.hf.encoder(input_ids=x[:, j], attention_mask=mask[:, j]).last_hidden_state for j in range(c)]
        return BaseModelOutput(last_hidden_state=torch.cat(states, dim=1)), mask.reshape(b, c * l)
    def loss(self, encoded, mask, y):
        dec = self.hf.prepare_decoder_input_ids_from_labels(y)
        dm = dec.ne(PAD_L).long(); dm[:, 0] = 1
        return self.hf(encoder_outputs=encoded, attention_mask=mask, decoder_input_ids=dec,
                       decoder_attention_mask=dm, labels=y, use_cache=False).loss
    @torch.no_grad()
    def generate(self, enc, am):
        return self.hf.generate(encoder_outputs=enc, attention_mask=am, max_new_tokens=MAX_NEW_TOKENS,
            min_new_tokens=MIN_NEW_TOKENS, num_beams=BEAMS, length_penalty=LENGTH_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM, early_stopping=BEAMS > 1, use_cache=True)

### 7c. Model factory (all four modes, both architectures)

In [ ]:
def pick_layers(layers):
    chosen = np.linspace(0, len(layers) - 1, STUDENT_LAYERS).round().astype(int)
    return nn.ModuleList([layers[int(i)] for i in chosen])

def make_model(method, student=False):
    seed_all()
    if ARCH == 'bert':
        hf = EncoderDecoderModel(encoder=build_backbone(), decoder=build_m7_decoder())
        if student:
            for bert in (hf.encoder, hf.decoder.bert):
                bert.encoder.layer = pick_layers(bert.encoder.layer)
                # Re-index attention modules so KV-cache slots match the shortened stack.
                for new_i, layer in enumerate(bert.encoder.layer):
                    for mod in layer.modules():
                        if hasattr(mod, 'layer_idx'): mod.layer_idx = new_i
                bert.config.num_hidden_layers = STUDENT_LAYERS
            hf.config.encoder.num_hidden_layers = STUDENT_LAYERS
            hf.config.decoder.num_hidden_layers = STUDENT_LAYERS
        if method in ('lora', 'lorma'):
            for p in hf.parameters(): p.requires_grad_(False)
            add_adapters(hf.encoder, method); add_adapters(hf.decoder.bert, method)
            for n, p in hf.decoder.named_parameters():
                if 'crossattention' in n or n.startswith('cls.'): p.requires_grad_(True)
        hf.config.decoder_start_token_id = START_ID
        hf.config.eos_token_id = EOS_ID
        hf.config.pad_token_id = PAD_L
        hf.config.vocab_size = hf.config.decoder.vocab_size
        for key in ('decoder_start_token_id', 'eos_token_id', 'pad_token_id'):
            setattr(hf.generation_config, key, getattr(hf.config, key))
        hf.decoder.config.use_cache = False
        if GRAD_CHECKPOINT:
            hf.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        model = SummBert(hf)
    else:
        enc_b, dec_b = build_backbone(), build_backbone()      # two copies of the same pre-trained model
        if student:
            for b in (enc_b, dec_b):
                b.layers = pick_layers(b.layers); b.n_layers = STUDENT_LAYERS
        model = SummLM(enc_b, dec_b, GRAD_CHECKPOINT)           # new cross-attention is created here (random init)
        if method in ('lora', 'lorma'):
            for p in model.parameters(): p.requires_grad_(False)
            add_adapters_lm(model.encoder.backbone, method); add_adapters_lm(model.decoder.backbone, method)
            for n, p in model.decoder.named_parameters():
                if 'xattn' in n: p.requires_grad_(True)
            model.decoder.backbone.tok_emb.weight.requires_grad_(True)   # tied decoder LM head
    print(MODEL_NAME, method, 'student', student, 'trainable', sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model.to(DEVICE)

@torch.no_grad()
def generate(model, ds):
    model.eval(); outputs = []
    for x, m, rows in tqdm(loader(ds), desc='Generate', leave=False):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            enc, am = model.encode(x, m)
            ids = model.generate(enc, am)
        texts = tokenizer.batch_decode(ids, skip_special_tokens=True)
        assert all(t.strip() for t in texts), 'Empty generated summary; inspect generation settings'
        outputs.extend(t.strip() for t in texts)
    return outputs

def trainable_state(model):
    # Everything else is identical to the pre-trained checkpoint (or is re-derived from it), which make_model() reloads.
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

## 8. Validation and training
Checkpoint selection uses generated-summary BERTScore F1, with ROUGE-1/L F1 also reported. BERTScore runs on CPU to leave GPU memory for training. This is a local proxy, not an assertion that competition scores will match. Every epoch evaluates the complete held-out split; this can be slow.

In [ ]:
def metrics(preds):
    refs=[r['summary'] for r in dev]
    scorer=rouge_scorer.RougeScorer(['rouge1','rougeL'],use_stemmer=True)
    rs=[scorer.score(r,p) for r,p in zip(refs,preds)]
    _,_,f=bert_score(preds,refs,model_type=BERTSCORE_MODEL,num_layers=BERTSCORE_LAYERS,
        device='cpu',batch_size=8,verbose=False,rescale_with_baseline=False)
    return {'bertscore_f1':float(f.mean()),**{k:float(np.mean([s[k].fmeasure for s in rs])) for k in ('rouge1','rougeL')}}

def fit(method,name,student=False,distill=False):
    model=make_model(method,student)
    groups={ 'base':[], 'new':[], 'adapter':[] }
    for n,p in model.named_parameters():
        if p.requires_grad:
            key='adapter' if n.endswith(('.A','.B')) else ('new' if 'crossattention' in n or 'xattn' in n or 'decoder.cls.' in n else 'base')
            groups[key].append(p)
    opt=torch.optim.AdamW([{'params':ps,'lr':{'base':FULL_LR,'new':NEW_LR,'adapter':ADAPTER_LR}[k]} for k,ps in groups.items() if ps],weight_decay=WEIGHT_DECAY)
    batches=loader(train_ds,True); steps=math.ceil(len(batches)/GRAD_ACCUM)*EPOCHS
    sched=get_linear_schedule_with_warmup(opt,int(steps*WARMUP_FRAC),steps)
    scaler=torch.amp.GradScaler('cuda',enabled=AMP)
    best=-float('inf'); bad=0; history=[]; path=OUTPUT_DIR/f'best_{name}.pt'
    for epoch in range(EPOCHS):
        model.train(); opt.zero_grad(set_to_none=True); losses=[]
        for i,(x,m,rows) in enumerate(tqdm(batches,desc=f'{name} epoch {epoch+1}')):
            with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=AMP):
                enc,am=encode(model,x,m)
                loss=model.loss(enc,am,labels_for(rows))
                if distill:
                    soft=model.loss(enc,am,labels_for(rows,'teacher_summary'))
                    loss=KD_HARD_WEIGHT*loss+(1-KD_HARD_WEIGHT)*soft
            assert torch.isfinite(loss), 'Non-finite loss'
            size=min(GRAD_ACCUM,len(batches)-(i//GRAD_ACCUM)*GRAD_ACCUM)
            scaler.scale(loss/size).backward(); losses.append(loss.item())
            if (i+1)%GRAD_ACCUM==0 or i+1==len(batches):
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.)
                old_scale=scaler.get_scale(); scaler.step(opt); scaler.update()
                if scaler.get_scale()>=old_scale: sched.step()
                opt.zero_grad(set_to_none=True)
        scores=metrics(generate(model,dev_ds))
        row={'epoch':epoch+1,'train_loss':float(np.mean(losses)),**scores}; history.append(row); print(row)
        if scores['bertscore_f1']>best:
            best=scores['bertscore_f1']; bad=0
            torch.save({'state':trainable_state(model),'model_option':MODEL_OPTION,
                'method':method,'student':student,'metrics':scores},path)
        else:
            bad+=1
            if bad>=PATIENCE: break
    with open(OUTPUT_DIR/f'history_{name}.json','w') as f: json.dump(history,f,indent=2)
    del model,opt,sched,scaler; cleanup()
    return path

def restore(path):
    ck=torch.load(path,map_location='cpu',weights_only=True)
    assert ck['model_option']==MODEL_OPTION, f"checkpoint is for Model {ck['model_option']}, notebook is set to Model {MODEL_OPTION}"
    model=make_model(ck['method'],ck['student'])
    missing,unexpected=model.load_state_dict(ck['state'],strict=False)
    assert not unexpected, unexpected[:5]
    expected={n for n,p in model.named_parameters() if p.requires_grad}
    assert expected<=set(ck['state']), list(expected-set(ck['state']))[:5]
    return model,ck['metrics']

## 9. Run modes and sequence-level knowledge distillation
Selecting knowledge_distillation automatically trains a full teacher if this run has not trained one. The teacher sees only training references and generates targets only for training documents. The student keeps `STUDENT_LAYERS` evenly spaced layers of the pre-trained model in both encoder and decoder, then learns from both original references and teacher sequences. Validation always uses original references. SMOKE is only a pipeline check and produces an incomplete, non-submittable test output.

In [ ]:
config={k:v for k,v in list(globals().items()) if k.isupper() and isinstance(v,(str,int,float,bool,list,type(None))) and 'TOKEN' not in k}
with open(OUTPUT_DIR/'config.json','w') as f: json.dump(config,f,indent=2)
tokenizer.save_pretrained(OUTPUT_DIR/'tokenizer')
results={}; teacher_path=None
for method in RUN_METHODS:
    if method=='knowledge_distillation':
        if teacher_path is None: teacher_path=fit('full','kd_teacher')
        teacher,_=restore(teacher_path)
        pseudo=generate(teacher,train_ds)
        for r,s in zip(train,pseudo): r['teacher_summary']=s
        with open(OUTPUT_DIR/'teacher_training_summaries.json','w',encoding='utf-8') as f:
            json.dump([{'id':r['id'],'summary':s} for r,s in zip(train,pseudo)],f,ensure_ascii=False)
        del teacher; cleanup()
        path=fit('full',method,student=True,distill=True)
    else:
        path=fit(method,method)
        if method=='full': teacher_path=path
    model,scores=restore(path)
    preds=generate(model,test_ds)
    submission=[{'id':r['id'],'summary':s} for r,s in zip(test,preds)]
    assert len(submission)==len(test) and [r['id'] for r in submission]==[r['id'] for r in test]
    assert all(set(r)=={'id','summary'} and r['summary'].strip() for r in submission)
    outfile=OUTPUT_DIR/f'prediction_{method}.json'
    with open(outfile,'w',encoding='utf-8') as f: json.dump(submission,f,ensure_ascii=False,indent=2)
    results[method]={'checkpoint':str(path),'prediction':str(outfile),**scores}
    del model; cleanup()
winner=max(results,key=lambda k:results[k]['bertscore_f1'])
final=OUTPUT_DIR/('smoke_prediction.json' if SMOKE else 'prediction.json')
shutil.copyfile(results[winner]['prediction'],final)
with open(OUTPUT_DIR/'comparison.json','w') as f: json.dump(results,f,indent=2)
if not SMOKE:
    with zipfile.ZipFile(OUTPUT_DIR/'submission.zip','w',zipfile.ZIP_DEFLATED) as z: z.write(final,arcname='prediction.json')
print('Selected by validation BERTScore:',winner,'Saved:',final)
from IPython.display import display, FileLink
display(FileLink(str(final)))

## 10. Practical notes and experiments
- Start with a LoRA/LoRMA run. Full training and the distillation teacher need more memory and time: this notebook holds **two copies** of the selected model (encoder + decoder) plus the cross-attention, and Model 6 is the heaviest. For GPU OOM reduce CHUNKS, MAX_LEN, MAX_TARGET and BEAMS; BATCH_SIZE is already 1. `GRAD_CHECKPOINT=True` is on for every mode. No GPU runtime or quality guarantee has been measured here.
- Run the notebook once per model (`MODEL_OPTION = 4`, `6`, `7`) with the same split/seed to compare them; each run writes to its own `summ_p6_model<N>` folder.
- Compare uniform and head_tail on the same saved split, then increase the source budget. Token states preserve more information than one vector per window, but decoder cross-attention grows with the total selected tokens. Selected windows are not exhaustive coverage and this is not recursive map/reduce summarization.
- Compare all four modes with `RUN_METHODS=['lora','lorma','full','knowledge_distillation']`; this trains several large models sequentially, so allow sufficient disk and session time.
- Check factual consistency, final disposition, party names and citations on held-out documents. BERTScore/ROUGE alone do not guarantee factual preservation.
- Restore in another session by running the definitions with the same config (model option, rank, alpha, student depth), then `model, scores = restore(Path(...))`. Checkpoints hold the trainable tensors on top of your pre-trained checkpoint, so that checkpoint and tokenizer must be attached again. They are inference checkpoints, not optimizer-resume checkpoints.
- Models 4/6 use a custom beam search (same settings as the original generation config: beams, length penalty, no-repeat n-gram, min/max new tokens, early stopping); it is not byte-identical to Hugging Face's implementation.

References: [official dataset](https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone), [Transformers encoder-decoder API](https://huggingface.co/docs/transformers/v4.44.2/en/model_doc/encoder-decoder). Dataset access and package compatibility are checked only when you run on Kaggle.